#CONFIG

In [0]:
%sql
DROP TABLE IF EXISTS la_lakehouse.bronze.la_parcels;
DROP TABLE IF EXISTS la_lakehouse.bronze.la_building_permits;


In [0]:
INTERNAL_CONFIG = [
    {
        "API_URL": "https://data.lacity.org/resource/qyra-qm2s.csv",
        "DELTA_TABLE_NAME": "la_lakehouse.bronze.la_parcels",
        "PARAMS" : {
            "$limit" : 10000,
            "$offset" : 0,
            "$order" : "id",
            "$select": "*,:updated_at" 
        }
    },
    {
        "API_URL": "https://data.lacity.org/resource/pi9x-tg5x.csv",
        "DELTA_TABLE_NAME": "la_lakehouse.bronze.la_building_permits_issued",
        "PARAMS" : {
            "$limit" : 10000,
            "$offset" : 0,
            "$order" : "permit_nbr",
            "$select": "*,:updated_at" 
        }
    }
]

#INIT


In [0]:
import pandas as pd
import requests
import io
from pyspark.sql.types import StructType, StructField, StringType


#Pulling From API And Writting to Bronze Table

In [0]:
APP_TOKEN = dbutils.secrets.get(scope="la-lakehouse", key="socrata-app-token")
headers = {
        "X-App-Token": APP_TOKEN
    }

for config in INTERNAL_CONFIG:
    API_URL = config["API_URL"]
    DELTA_TABLE_NAME = config["DELTA_TABLE_NAME"]
    limit = config['PARAMS']["$limit"]
    offset = config['PARAMS']["$offset"]


    print(f"Loading {DELTA_TABLE_NAME}...")

    is_first_chunk = True
    table_schema = None

    while True:
        print(f"Fetching rows {offset} to {offset + limit}...")

        response = requests.get(API_URL, headers=headers, params=config['PARAMS'])

        if response.status_code != 200:
            print(f"API Error! {response.status_code}: {response.text}")
            break

        df_chunk = pd.read_csv(io.StringIO(response.text), low_memory=False, dtype=str)
        
        if df_chunk.empty:
            break
        df_chunk = df_chunk.rename(columns={":updated_at": "_updated_at"})

        if table_schema is None:
            table_schema = StructType([StructField(col, StringType(), True) for col in df_chunk.columns])

        spark_chunk_df = spark.createDataFrame(df_chunk, schema=table_schema)

        if is_first_chunk:
            spark_chunk_df.write.mode("overwrite").format('delta').saveAsTable(DELTA_TABLE_NAME) 
            is_first_chunk = False
        else:
            spark_chunk_df.write.mode("append").format('delta').saveAsTable(DELTA_TABLE_NAME)

        offset += limit
        config['PARAMS']['$offset'] = offset

        if len(df_chunk) < limit:
            break

    print(f"Finished processing and fully updated: {DELTA_TABLE_NAME}\n")

#TESTING

In [0]:
%sql
SELECT COUNT(*) FROM la_lakehouse.bronze.la_parcels;

In [0]:
%sql
SELECT ASSETID, COUNT(*) FROM la_lakehouse.bronze.la_parcels GROUP BY ASSETID HAVING COUNT(*) > 1 LIMIT 20;

In [0]:
%sql
SELECT COUNT(*) FROM la_lakehouse.bronze.la_building_permits_issued;

In [0]:
%sql
SELECT permit_nbr, COUNT(*) FROM la_lakehouse.bronze.la_building_permits_issued GROUP BY permit_nbr HAVING COUNT(*) > 1 LIMIT 20;

In [0]:
%sql

SELECT *
FROM la_lakehouse.bronze.la_parcels
LIMIT 10;


In [0]:
%sql
SELECT * 
FROM la_lakehouse.bronze.la_building_permits_issued
LIMIT 10;

In [0]:
%sql
DESCRIBE la_lakehouse.bronze.la_parcels;

In [0]:
%sql
DESCRIBE la_lakehouse.bronze.la_building_permits_issued;